# 📊 Exploratory Data Analysis (EDA) — Email Risk Analyzer
This notebook performs detailed Exploratory Data Analysis (EDA) on the standardized email classification dataset. We will explore:
1. Class distributions
2. Email lengths and word count distributions
3. Key metadata features (uppercase ratio, special character counts)
4. Top vocabulary and n-grams per email class
5. Word clouds visualizing spam, phishing, safe, and promotional signatures

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import sys

# Ensure parent directory is in sys.path to import utils
sys.path.insert(0, os.path.abspath('..'))
from utils.preprocessing import clean_text, extract_meta_features

## 1. Load the Dataset

In [ ]:
DATA_PATH = '../data/processed/emails_dataset.csv'
if not os.path.exists(DATA_PATH):
    # Fallback to current directory path if running from notebooks/ dir
    DATA_PATH = 'data/processed/emails_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} emails.")
df.head()

## 2. Class Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='label', palette='viridis', order=df['label'].value_counts().index)
plt.title('Email Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 3. Metadata Feature Extraction
Let's extract metadata features using our preprocessing module to see if they differ across classes.

In [ ]:
print("Extracting metadata features...")
meta_list = [extract_meta_features(t) for t in df['text']]
df_meta = pd.DataFrame(meta_list)
df_meta['label'] = df['label']
df_meta.head()

### 3.1. Email Length Distribution by Category

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_meta, x='label', y='email_length', palette='Set2')
plt.title('Email Length Distribution by Category (Log Scale)', fontsize=14, fontweight='bold')
plt.yscale('log')
plt.xlabel('Category')
plt.ylabel('Length (characters)')
plt.show()

### 3.2. Uppercase Letter Ratio by Category

In [ ]:
plt.figure(figsize=(12, 6))
sns.violinplot(data=df_meta, x='label', y='capital_ratio', palette='muted')
plt.title('Capitalization Ratio by Category', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Capital Letter Ratio')
plt.show()

### 3.3. URL Frequencies across Classes

In [ ]:
url_summary = df_meta.groupby('label')['url_count'].mean().reset_index()
plt.figure(figsize=(8, 5))
sns.barplot(data=url_summary, x='label', y='url_count', palette='coolwarm')
plt.title('Average Number of URLs per Email', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Avg URLs')
plt.show()

## 4. Word Clouds per Class

In [ ]:
categories = df['label'].unique()
fig, axes = plt.subplots(3, 2, figsize=(16, 20))
axes = axes.flatten()

for i, cat in enumerate(categories):
    cat_texts = df[df['label'] == cat]['text'].astype(str)
    cleaned_texts = " ".join([clean_text(t) for t in cat_texts])
    
    wordcloud = WordCloud(width=800, height=400, background_color='black',
                          colormap='viridis', max_words=100).generate(cleaned_texts)
    
    axes[i].imshow(wordcloud, interpolation='bilinear')
    axes[i].set_title(f'Word Cloud: {cat.upper()}', fontsize=16, fontweight='bold')
    axes[i].axis('off')
    
# Hide the last subplot since we only have 5 categories
axes[-1].axis('off')
plt.tight_layout()
plt.show()

## 5. Most Frequent Words per Class

In [ ]:
for cat in categories:
    cat_texts = df[df['label'] == cat]['text'].astype(str)
    cleaned_words = []
    for text in cat_texts:
        cleaned_words.extend(clean_text(text).split())
        
    counter = Counter(cleaned_words)
    common_words = counter.most_common(15)
    
    words, counts = zip(*common_words)
    plt.figure(figsize=(10, 4))
    sns.barplot(x=list(counts), y=list(words), palette='rocket')
    plt.title(f'Top 15 Most Frequent Words in {cat.upper()} Emails', fontsize=12, fontweight='bold')
    plt.xlabel('Frequency')
    plt.ylabel('Word')
    plt.show()